In [ ]:
# ── CELL 0: Imports ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
np.random.seed(42)
print('Ready.')

In [ ]:
# ── CELL 1: The Core Loop — What "Learning" Actually Means ─────────────────────
#
# Every ML model — from linear regression to GPT-4 — learns via ONE loop:
#
#   1. FORWARD PASS:   make a prediction with current parameters
#   2. LOSS:           measure how wrong the prediction is (a single number)
#   3. BACKWARD PASS:  compute how much each parameter contributed to that error
#   4. UPDATE:         nudge each parameter in the direction that reduces error
#   5. REPEAT:         thousands to billions of times
#
# The parameters are the model's "weights" — just numbers.
# Learning = iteratively adjusting those numbers to reduce the loss.
#
# ANALOGY (backend engineering)
# You have a service that responds to requests. You have a performance target
# (SLA: p99 < 100ms). You measure the gap (loss). You tune config params
# (weights) to close that gap. You monitor and iterate.
# ML training is the same loop, but automated and mathematical.

# Concrete minimal example — one parameter, one equation
# Problem: learn y = 3x from data

np.random.seed(42)
X = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y = 3.0 * X   # true relationship: slope = 3

w = 0.0   # our model's single parameter — start at 0 (wrong)
lr = 0.01 # learning rate — how big a step to take each update

print(f'True slope: 3.0')
print(f'Starting w: {w}')
print()
print(f'{"Step":>5} | {"w":>8} | {"Loss (MSE)":>12} | {"Gradient":>10}')
print('-' * 50)

for step in range(1, 51):
    y_pred   = w * X                         # forward pass
    loss     = np.mean((y_pred - y) ** 2)    # loss (MSE)
    grad     = np.mean(2 * (y_pred - y) * X) # gradient of loss w.r.t. w
    w        = w - lr * grad                 # update: move opposite to gradient

    if step in [1, 5, 10, 20, 30, 50]:
        print(f'{step:>5} | {w:>8.4f} | {loss:>12.4f} | {grad:>10.4f}')

print()
print(f'Learned w = {w:.4f}  (target = 3.0)')
print('The model learned y = 3x purely by reducing its own error.')

In [ ]:
# ── CELL 2: Loss Functions — The Scorecard ─────────────────────────────────────
#
# The loss function is the ONLY signal the model gets about whether it's
# improving. Choose the wrong one and the model optimizes the wrong thing.
#
# This is one of the most consequential decisions in applied ML.
#
# ── REGRESSION LOSSES ──────────────────────────────────────────────────────────
#
# MSE — Mean Squared Error:  mean((y_pred - y)²)
#   → Large errors penalized heavily (squared)
#   → Sensitive to outliers — one bad prediction dominates
#   → Use when: outliers are rare and genuinely signal a problem
#   → Common in: house price prediction, demand forecasting
#
# MAE — Mean Absolute Error: mean(|y_pred - y|)
#   → All errors treated equally (linear)
#   → Robust to outliers
#   → Use when: outliers exist and should not dominate training
#   → Common in: delivery time, rental price prediction
#
# Huber Loss:
#   → MSE when error is small, MAE when error is large
#   → Best of both worlds — smooth + outlier robust
#   → Use when: mixed data quality, real-world sensor data
#
# ── CLASSIFICATION LOSSES ──────────────────────────────────────────────────────
#
# Binary Cross-Entropy: -[y·log(p) + (1-y)·log(1-p)]
#   → The standard loss for binary classification (spam/not-spam)
#   → Penalizes CONFIDENT wrong answers heavily (log scale)
#   → Use when: output is a probability (0 to 1)
#
# Categorical Cross-Entropy: -sum(y_true · log(y_pred))
#   → Multi-class classification (one of N classes)
#   → Used with softmax output layer in neural networks
#
# ── PRODUCTION REALITY ─────────────────────────────────────────────────────────
# The loss function you train with is NOT always what the business cares about.
#
# Example: fraud detection
#   Train loss:      binary cross-entropy (standard)
#   Business metric: recall@precision=0.95 (catch 95% of fraud)
#   These are DIFFERENT. The model optimizes cross-entropy, you monitor recall.
#
# Common mismatch patterns in production:
#   - Train on MSE, business cares about MAPE (percentage error)
#   - Train on accuracy, business cares about F1 (imbalanced classes)
#   - Train on log-loss, business cares about ranking (AUC-ROC)
#
# → Always define the business metric FIRST, then pick the nearest proxy loss.

np.random.seed(42)
y_true = np.array([1.0, 2.0, 3.0, 10.0, 5.0])  # note: 10.0 is an outlier
y_pred = np.array([1.2, 1.8, 3.5,  4.0, 5.2])

mse   = np.mean((y_pred - y_true) ** 2)
mae   = np.mean(np.abs(y_pred - y_true))
delta = 1.0
huber = np.mean(np.where(np.abs(y_pred - y_true) <= delta,
                          0.5 * (y_pred - y_true)**2,
                          delta * np.abs(y_pred - y_true) - 0.5 * delta**2))

print('Regression Loss Comparison (with one outlier y=10, pred=4):')
print(f'  MSE:   {mse:.3f}  ← dominated by the outlier (10→4 = error of 6²=36)')
print(f'  MAE:   {mae:.3f}  ← outlier treated same as small errors')
print(f'  Huber: {huber:.3f} ← compromise: outlier capped at linear penalty')
print()

# Cross-entropy: penalizes confident wrong predictions harshly
eps = 1e-9
scenarios = [
    ('Correctly confident   (true=1, pred=0.95)', 1, 0.95),
    ('Correctly uncertain   (true=1, pred=0.60)', 1, 0.60),
    ('Wrong but uncertain   (true=1, pred=0.40)', 1, 0.40),
    ('Confidently wrong     (true=1, pred=0.05)', 1, 0.05),
]
print('Binary Cross-Entropy — why confident wrong predictions are punished most:')
for label, y, p in scenarios:
    bce = -(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
    bar = '█' * int(bce * 10)
    print(f'  {label}: BCE = {bce:.3f}  {bar}')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

errors = np.linspace(-3, 3, 300)
axes[0].plot(errors, errors**2,                     lw=2, label='MSE')
axes[0].plot(errors, np.abs(errors),                lw=2, label='MAE')
axes[0].plot(errors, np.where(np.abs(errors) <= 1,
             0.5*errors**2, np.abs(errors) - 0.5),   lw=2, label='Huber (δ=1)')
axes[0].set_title('Regression Losses — How Each Penalizes Errors')
axes[0].set_xlabel('Prediction Error'); axes[0].legend()

p_vals = np.linspace(0.001, 0.999, 300)
axes[1].plot(p_vals, -np.log(p_vals),       lw=2, label='BCE when y=1 (want p→1)')
axes[1].plot(p_vals, -np.log(1 - p_vals),   lw=2, label='BCE when y=0 (want p→0)')
axes[1].set_title('Binary Cross-Entropy — Confident Wrong = High Loss')
axes[1].set_xlabel('Predicted Probability'); axes[1].set_ylim(0, 5); axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 3: Gradients — The Compass That Points Downhill ───────────────────────
#
# A gradient is just the slope of the loss function at the current parameter
# values. It tells you: "if I increase this parameter, does the loss go up
# or down, and by how much?"
#
# Imagine the loss function as a mountain landscape.
# Your model parameters are your position on that landscape.
# The gradient is the slope beneath your feet.
# You want to walk DOWNHILL (to the minimum loss).
# The gradient tells you which direction is downhill.
#
# For a single parameter w, loss L:
#   gradient = dL/dw  (partial derivative)
#   new_w    = w - learning_rate * gradient
#
# The minus sign: gradient points UPHILL, so we go in the opposite direction.
#
# For neural networks with millions of parameters:
#   - Gradient is a vector of the same shape as all parameters
#   - Each element = how much this one weight contributed to the loss
#   - Computed via BACKPROPAGATION (covered in Stage 2)
#
# Numerical vs analytical gradient:
#   Numerical:  (L(w+ε) - L(w-ε)) / 2ε  — slow, used for testing only
#   Analytical: math formula              — fast, what frameworks use

# Visualize the loss landscape and gradient descent path
np.random.seed(42)

# Simple 2D loss surface: L(w) = (w - 3)²
w_vals = np.linspace(-1, 7, 300)
L_vals = (w_vals - 3) ** 2  # minimum at w=3

# Gradient descent trajectory
w_curr = -0.5  # start far from minimum
lr     = 0.15
path   = [w_curr]
for _ in range(20):
    grad   = 2 * (w_curr - 3)   # dL/dw = 2(w-3)
    w_curr = w_curr - lr * grad
    path.append(w_curr)

path = np.array(path)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: loss curve + descent path
axes[0].plot(w_vals, L_vals, 'steelblue', lw=2.5, label='Loss L(w) = (w-3)²')
axes[0].scatter(path, (path - 3)**2, c='tomato', s=60, zorder=5)
axes[0].plot(path, (path - 3)**2, 'tomato', lw=1.5, alpha=0.7, label='GD path')
axes[0].axvline(3, color='seagreen', ls='--', lw=1.5, label='Minimum (w=3)')
axes[0].set_title('Gradient Descent — Walking Downhill on the Loss Surface')
axes[0].set_xlabel('Parameter w'); axes[0].set_ylabel('Loss')
axes[0].legend()

# Right: gradient magnitude at each step
grads = 2 * (path - 3)
axes[1].bar(range(len(grads)), np.abs(grads), color='steelblue', alpha=0.7)
axes[1].set_title('Gradient Magnitude — Gets Smaller as We Approach Minimum')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('|Gradient|')

plt.tight_layout(); plt.show()

# Numerical vs analytical gradient check (the "gradient check" in practice)
def loss_fn(w): return (w - 3.0) ** 2
w_test = 1.0
eps    = 1e-5

grad_numerical  = (loss_fn(w_test + eps) - loss_fn(w_test - eps)) / (2 * eps)
grad_analytical = 2 * (w_test - 3.0)

print('Gradient check (used when debugging custom loss functions):')
print(f'  Numerical gradient:  {grad_numerical:.8f}')
print(f'  Analytical gradient: {grad_analytical:.8f}')
print(f'  Difference:          {abs(grad_numerical - grad_analytical):.2e}  ← should be ~1e-5 or smaller')
print()
print('In PyTorch/TensorFlow: torch.autograd.gradcheck() does this automatically.')
print('Run gradient checks when implementing custom loss functions in production.')

In [ ]:
# ── CELL 4: Gradient Descent Variants — The Ones That Ship ─────────────────────
#
# Pure gradient descent (Batch GD) is almost never used in production.
# Here are the three variants and when each is used:
#
# BATCH GRADIENT DESCENT
#   - Compute gradient on the ENTIRE dataset before each update
#   - Exact gradient, smooth convergence
#   - Impossible for large datasets (you can't fit 1TB of data in RAM)
#   - Never used in modern DL
#
# STOCHASTIC GRADIENT DESCENT (SGD)
#   - Compute gradient on ONE sample at a time
#   - Fast updates, noisy path
#   - Noise helps escape local minima (useful in deep learning)
#   - Used in: online learning, very large datasets
#
# MINI-BATCH GRADIENT DESCENT (the default in production)
#   - Compute gradient on a BATCH (32, 64, 128, 256 samples)
#   - Balance between stability and speed
#   - GPU-friendly: matrix ops on batches are highly parallelizable
#   - This is what every PyTorch/TensorFlow training loop uses
#
# PRODUCTION BATCH SIZE DECISIONS:
#   Larger batch → more stable gradients, but:
#     - Needs more GPU memory
#     - Can converge to sharper (worse generalizing) minima
#     - Common sizes: 32–512 for vision, 4–32 for LLMs (memory constrained)
#   Smaller batch → noisier, but sometimes generalizes better
#   Rule of thumb: start at 32, scale up only if training is slow

np.random.seed(42)
n = 500
X_data = np.random.randn(n)
y_data = 3.0 * X_data + 1.0 + np.random.randn(n) * 0.5  # y = 3x + 1 + noise

def run_gd_variant(variant, epochs=30, lr=0.05, batch_size=32):
    w, b = 0.0, 0.0
    loss_history = []

    for epoch in range(epochs):
        indices = np.random.permutation(n)  # shuffle each epoch
        epoch_loss = []

        if variant == 'batch':
            batches = [indices]  # one batch = full dataset
        elif variant == 'sgd':
            batches = [[i] for i in indices]  # one sample per batch
        else:  # mini-batch
            batches = [indices[i:i+batch_size] for i in range(0, n, batch_size)]

        for batch in batches:
            xb = X_data[batch]; yb = y_data[batch]
            yp = w * xb + b
            loss = np.mean((yp - yb) ** 2)
            epoch_loss.append(loss)
            gw = np.mean(2 * (yp - yb) * xb)
            gb = np.mean(2 * (yp - yb))
            w -= lr * gw
            b -= lr * gb

        loss_history.append(np.mean(epoch_loss))

    return loss_history, w, b

# SGD is very slow per epoch — limit it
batch_hist, bw, bb   = run_gd_variant('batch',    epochs=30)
mini_hist,  mw, mb   = run_gd_variant('mini',     epochs=30)
sgd_hist,   sw, sb   = run_gd_variant('sgd',      epochs=10, lr=0.01)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(batch_hist, lw=2, label='Batch GD (full dataset)')
axes[0].plot(mini_hist,  lw=2, label='Mini-batch GD (batch=32) ← default')
axes[0].plot(sgd_hist,   lw=2, label='SGD (1 sample)', alpha=0.7)
axes[0].set_title('Loss Convergence by GD Variant')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].legend(); axes[0].set_yscale('log')

# Smoothness comparison on same scale
axes[1].plot(mini_hist, lw=2, label='Mini-batch (smooth + fast)')
axes[1].plot(batch_hist, lw=2, ls='--', label='Batch (smoothest, slow/impractical)')
axes[1].set_title('Mini-batch vs Batch — Why Mini-batch Wins in Production')
axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.tight_layout(); plt.show()

print(f'All learned y = wx + b:')
print(f'  Batch GD:    w={bw:.3f} b={bb:.3f}')
print(f'  Mini-batch:  w={mw:.3f} b={mb:.3f}')
print(f'  True:        w=3.000 b=1.000')

In [ ]:
# ── CELL 5: The Learning Rate — The Most Important Hyperparameter ──────────────
#
# The learning rate (lr) controls the step size each update.
# It is the single most important hyperparameter to get right.
#
#   Too HIGH:  steps too large → overshoots minimum → loss oscillates or diverges
#   Too LOW:   steps too small → training takes forever → may get stuck
#   Just right: converges steadily to a good minimum
#
# PRODUCTION STARTING POINTS:
#   Scikit-learn models:  no lr to tune (uses closed-form or line search)
#   SGD-based models:     start at 0.01
#   Adam optimizer:       start at 1e-3 (0.001) — most common default
#   Fine-tuning LLMs:     1e-5 to 5e-5 (very small — preserve pretrained knowledge)
#   Training from scratch:2e-4 to 1e-3
#
# LEARNING RATE SCHEDULES (production must-know)
# Instead of a fixed lr, decay it over training:
#   Step decay:    halve lr every N epochs
#   Cosine decay:  smooth decay following cosine curve (default in modern DL)
#   Warmup:        start small, ramp up, then decay — critical for transformers
#   Cyclical LR:   oscillate between min/max — helps escape local minima
#
# LR FINDER (a tool you will actually use in production):
# Start with very small lr, increase exponentially each batch,
# plot lr vs loss, pick the lr just BEFORE the loss starts rising.
# Implemented in PyTorch Lightning, fastai — standard practice.

np.random.seed(42)
X_lr = np.random.randn(200)
y_lr = 2.0 * X_lr + np.random.randn(200) * 0.3

def train_with_lr(lr, steps=60):
    w = 0.0
    history = []
    for _ in range(steps):
        yp   = w * X_lr
        loss = np.mean((yp - y_lr) ** 2)
        grad = np.mean(2 * (yp - y_lr) * X_lr)
        w   -= lr * grad
        history.append(loss)
    return history

configs = [
    (0.001, 'LR=0.001 (too small — slow)',   'gray'),
    (0.05,  'LR=0.05  (good)',               'seagreen'),
    (0.4,   'LR=0.4   (too large — unstable)','tomato'),
    (1.0,   'LR=1.0   (diverges)',           'orange'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr_val, label, color in configs:
    hist = train_with_lr(lr_val)
    hist_clipped = np.clip(hist, 0, 20)
    axes[0].plot(hist_clipped, label=label, color=color, lw=2)

axes[0].set_title('Learning Rate Effect on Training')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss (clipped at 20)')
axes[0].legend(fontsize=9)

# LR schedule demo: cosine decay with warmup
total_steps  = 100
warmup_steps = 10
base_lr      = 0.1

steps = np.arange(total_steps)
warmup_lr = base_lr * (steps / warmup_steps)
cosine_lr = base_lr * 0.5 * (1 + np.cos(np.pi * (steps - warmup_steps) / (total_steps - warmup_steps)))
lr_schedule = np.where(steps < warmup_steps, warmup_lr, cosine_lr)

axes[1].plot(steps, lr_schedule, 'steelblue', lw=2.5)
axes[1].axvline(warmup_steps, color='tomato', ls='--', label=f'End of warmup ({warmup_steps} steps)')
axes[1].set_title('Warmup + Cosine Decay Schedule\n(default in transformer training)')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Learning Rate')
axes[1].legend()

plt.tight_layout(); plt.show()

print('LR Schedule summary:')
print(f'  Steps  0-{warmup_steps}: warmup  (lr ramps from 0 → {base_lr})')
print(f'  Steps {warmup_steps}-{total_steps}: cosine decay (lr decays smoothly to 0)')
print('→ Warmup prevents unstable gradients at the start of training.')
print('→ Cosine decay maintains momentum without over-shooting.')

In [ ]:
# ── CELL 6: Optimizers — Beyond Plain Gradient Descent ─────────────────────────
#
# Plain gradient descent (w -= lr * grad) works but is slow.
# Modern optimizers add momentum and adaptive learning rates.
#
# SGD WITH MOMENTUM:
#   Accumulates a "velocity" in the direction of consistent gradients.
#   Like a ball rolling downhill — gains speed on consistent slopes,
#   smooths out noisy updates.
#   velocity = momentum * velocity + gradient
#   w -= lr * velocity
#   → Good for CNNs when tuned. Requires careful lr + momentum tuning.
#
# ADAM (Adaptive Moment Estimation) — THE PRODUCTION DEFAULT:
#   Combines momentum (first moment) with adaptive lr per-parameter (second moment).
#   Each parameter gets its own effective learning rate.
#   Sparse features get larger updates. Frequent features get smaller updates.
#   Works well out of the box with lr=1e-3.
#   → Used in: LLMs, transformers, most deep learning production systems.
#
# ADAMW:
#   Adam + weight decay (L2 regularization done correctly).
#   The default for training and fine-tuning transformers.
#   → Use this unless you have a reason not to.
#
# RMSPROP:
#   Adaptive lr, precursor to Adam.
#   Still used in some RL contexts.
#
# PRODUCTION RULE OF THUMB:
#   → Start with AdamW, lr=1e-3
#   → If training is unstable: lower lr or add warmup
#   → If final accuracy plateaus: try SGD+momentum for fine-tuning
#   → Never use plain SGD without momentum for deep networks

np.random.seed(42)

# 2D loss surface — saddle point scenario (where momentum helps)
# L(w1, w2) = 0.1*w1^2 + 2*w2^2
# Elongated bowl — gradient descent takes tiny steps in w1, huge in w2

def loss_2d(w1, w2): return 0.1 * w1**2 + 2.0 * w2**2
def grad_2d(w1, w2): return np.array([0.2 * w1, 4.0 * w2])

def vanilla_gd(lr=0.1, steps=50):
    w = np.array([3.0, 2.5])
    path = [w.copy()]
    for _ in range(steps):
        w -= lr * grad_2d(*w)
        path.append(w.copy())
    return np.array(path)

def momentum_gd(lr=0.1, mom=0.9, steps=50):
    w = np.array([3.0, 2.5])
    v = np.zeros(2)
    path = [w.copy()]
    for _ in range(steps):
        v  = mom * v + grad_2d(*w)
        w -= lr * v
        path.append(w.copy())
    return np.array(path)

def adam_opt(lr=0.3, steps=50, beta1=0.9, beta2=0.999, eps=1e-8):
    w  = np.array([3.0, 2.5])
    m  = np.zeros(2); v = np.zeros(2)
    path = [w.copy()]
    for t in range(1, steps + 1):
        g  = grad_2d(*w)
        m  = beta1 * m + (1 - beta1) * g
        v  = beta2 * v + (1 - beta2) * g**2
        mh = m / (1 - beta1**t)
        vh = v / (1 - beta2**t)
        w -= lr * mh / (np.sqrt(vh) + eps)
        path.append(w.copy())
    return np.array(path)

gd_path  = vanilla_gd(lr=0.1)
mom_path = momentum_gd(lr=0.1)
adam_path = adam_opt(lr=0.3)

# Contour plot
w1r = np.linspace(-3.5, 3.5, 200)
w2r = np.linspace(-3.0, 3.0, 200)
W1, W2 = np.meshgrid(w1r, w2r)
Z = loss_2d(W1, W2)

plt.figure(figsize=(10, 7))
plt.contourf(W1, W2, Z, levels=30, cmap='Blues', alpha=0.6)
plt.contour(W1, W2, Z, levels=30, colors='white', linewidths=0.4, alpha=0.5)

plt.plot(gd_path[:,0],   gd_path[:,1],   'tomato',   lw=2, marker='.',  ms=4, label='Vanilla GD')
plt.plot(mom_path[:,0],  mom_path[:,1],  'orange',   lw=2, marker='.',  ms=4, label='Momentum')
plt.plot(adam_path[:,0], adam_path[:,1], 'seagreen', lw=2, marker='.',  ms=4, label='Adam')
plt.scatter([0], [0], c='white', s=200, zorder=5, marker='*', label='Minimum')

plt.title('Optimizer Paths on Elongated Loss Surface\n(Adam reaches minimum fastest)')
plt.xlabel('w1'); plt.ylabel('w2')
plt.legend(loc='upper right')
plt.tight_layout(); plt.show()

print('Steps to reach near-minimum (loss < 0.05):')
for name, path in [('Vanilla GD', gd_path), ('Momentum', mom_path), ('Adam', adam_path)]:
    losses = [loss_2d(*p) for p in path]
    reached = next((i for i, l in enumerate(losses) if l < 0.05), None)
    print(f'  {name:12s}: {reached if reached else "not reached"} steps')

In [ ]:
# ── CELL 7: The Full Training Loop — Production Shape ──────────────────────────
#
# Now let's put it all together into a training loop that looks like
# what you'd actually write in a production ML pipeline.
#
# Structure of every training loop:
#
#   for epoch in range(num_epochs):
#       shuffle data
#       for batch in data:
#           forward pass → loss
#           backward pass → gradients
#           optimizer step → update weights
#       evaluate on validation set
#       log metrics
#       checkpoint model if best so far
#       early stop if no improvement
#
# PRODUCTION ADDITIONS vs textbook:
#   1. Validation set monitoring — not just train loss
#   2. Early stopping — stop when val loss stops improving (save compute + avoid overfit)
#   3. Model checkpointing — save best model weights, not the last ones
#   4. Metric logging — to MLflow / W&B / Azure ML (covered in Stage 7)
#   5. Gradient clipping — cap gradient magnitude to prevent exploding gradients
#      (critical for RNNs and LLMs: torch.nn.utils.clip_grad_norm_)
#
# EARLY STOPPING is your free anti-overfitting tool:
#   Monitor val_loss. If it doesn't improve for N epochs (patience), stop.
#   Save the model from the BEST val_loss epoch, not the last one.
#   → Saves GPU hours. Prevents shipping an overfit model.

np.random.seed(42)

# Synthetic dataset: nonlinear regression
n = 600
X_full = np.random.uniform(-3, 3, n)
y_full = np.sin(X_full) + 0.5 * X_full + np.random.randn(n) * 0.3

# Train/val/test split — always 3-way in production
X_tr, X_val, X_te = X_full[:400], X_full[400:500], X_full[500:]
y_tr, y_val, y_te = y_full[:400], y_full[400:500], y_full[500:]

# Simple polynomial feature model
def poly_features(x, degree=5):
    return np.column_stack([x**d for d in range(1, degree+1)])

# Normalize
X_tr_f  = poly_features(X_tr)
X_val_f = poly_features(X_val)
X_te_f  = poly_features(X_te)
mu, sigma = X_tr_f.mean(0), X_tr_f.std(0) + 1e-8
X_tr_f  = (X_tr_f  - mu) / sigma
X_val_f = (X_val_f - mu) / sigma
X_te_f  = (X_te_f  - mu) / sigma

# Manually implement a mini training loop
np.random.seed(0)
w = np.random.randn(5) * 0.01
b = 0.0
lr = 0.05
batch_size = 32

train_losses, val_losses = [], []
best_val_loss  = np.inf
best_weights   = (w.copy(), b)
patience       = 10
patience_count = 0

for epoch in range(200):
    # Shuffle
    idx = np.random.permutation(len(X_tr_f))
    epoch_loss = []

    for i in range(0, len(X_tr_f), batch_size):
        xb = X_tr_f[idx[i:i+batch_size]]
        yb = y_tr[idx[i:i+batch_size]]
        yp = xb @ w + b
        loss = np.mean((yp - yb)**2)
        epoch_loss.append(loss)
        gw = (2/len(xb)) * xb.T @ (yp - yb)
        gb = (2/len(xb)) * np.sum(yp - yb)
        w -= lr * gw
        b -= lr * gb

    # Validation
    yp_val   = X_val_f @ w + b
    val_loss = np.mean((yp_val - y_val)**2)
    train_losses.append(np.mean(epoch_loss))
    val_losses.append(val_loss)

    # Checkpoint
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_weights   = (w.copy(), b)
        patience_count = 0
    else:
        patience_count += 1

    # Early stopping
    if patience_count >= patience:
        print(f'Early stopping at epoch {epoch} (best val_loss={best_val_loss:.4f})')
        break

# Evaluate best model on test set
w_best, b_best = best_weights
yp_te   = X_te_f @ w_best + b_best
test_loss = np.mean((yp_te - y_te)**2)

print(f'Final test loss (best checkpoint): {test_loss:.4f}')

plt.figure(figsize=(10, 4))
plt.plot(train_losses, 'steelblue', lw=2, label='Train loss')
plt.plot(val_losses,   'tomato',    lw=2, label='Val loss')
plt.axvline(np.argmin(val_losses), color='seagreen', ls='--',
            label=f'Best val (epoch {np.argmin(val_losses)})')
plt.title('Training Loop with Early Stopping & Checkpointing')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.legend(); plt.tight_layout(); plt.show()

print()
print('→ Best model is from epoch', np.argmin(val_losses), '— not the last epoch.')
print('→ In production: always checkpoint by val_loss, not by epoch number.')

In [ ]:
# ── CELL 8: Common Training Failures & Diagnostics ─────────────────────────────
#
# This is what separates engineers who can debug production training runs
# from engineers who can only run tutorials.
#
# ── SYMPTOM: Loss is NaN or Inf ────────────────────────────────────────────────
#   Cause:    exploding gradients, bad learning rate, bad data (inf/nan values)
#   Fix:      gradient clipping, lower lr, check input data for nan/inf
#   Code:     torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#
# ── SYMPTOM: Loss not decreasing at all ────────────────────────────────────────
#   Cause:    lr too small, wrong loss function, bug in forward pass, dead ReLU
#   Fix:      check gradient magnitudes, try 10x lr, verify loss fn is correct
#   Check:    assert loss.item() != 0 before training loop
#
# ── SYMPTOM: Train loss low, val loss high (overfitting) ───────────────────────
#   Cause:    model too large for dataset, no regularization
#   Fix:      dropout, weight decay, more data, early stopping
#   Monitor:  val_loss / train_loss ratio — should stay near 1.0
#
# ── SYMPTOM: Train and val loss both high (underfitting) ───────────────────────
#   Cause:    model too simple, too few epochs, bad features
#   Fix:      larger model, more epochs, better feature engineering
#
# ── SYMPTOM: Loss oscillates wildly ───────────────────────────────────────────
#   Cause:    lr too large
#   Fix:      lower lr by 10x, add warmup
#
# ── SYMPTOM: Loss decreases then suddenly spikes up ───────────────────────────
#   Cause:    bad batch (corrupt data), lr too high, or a single large-gradient sample
#   Fix:      gradient clipping, data validation pipeline, Huber loss
#
# PRODUCTION MONITORING CHECKLIST (your Grafana dashboard for ML):
#   □ train_loss per step
#   □ val_loss per epoch
#   □ gradient norm (should be ~0.1–10, not 0 or 1000+)
#   □ learning rate (confirm schedule is working)
#   □ GPU utilization (should be >80% — if lower, data loading is the bottleneck)
#   □ GPU memory (watch for OOM before they crash your job)

# Simulate and diagnose the three most common failure modes
np.random.seed(42)
steps = 60
base_loss = np.exp(-np.linspace(0, 3, steps)) * 2 + 0.1

# Healthy
healthy     = base_loss + np.random.randn(steps) * 0.05
# LR too high — oscillating
oscillating = base_loss + np.random.randn(steps) * 0.6 + np.sin(np.linspace(0, 8, steps)) * 0.5
oscillating = np.abs(oscillating)
# Exploding gradients — spikes then NaN
exploding   = base_loss.copy()
exploding[20] = 15.0; exploding[21] = 45.0; exploding[22:] = np.nan
# Underfitting — loss stops high
underfitting = 1.8 * np.ones(steps) + np.random.randn(steps) * 0.05

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
configs = [
    (healthy,      'Healthy — steady decay',             'seagreen'),
    (oscillating,  'LR too high — oscillating',         'orange'),
    (exploding,    'Exploding gradients → NaN crash',   'tomato'),
    (underfitting, 'Underfitting — loss stuck high',     'steelblue'),
]
for ax, (data, title, color) in zip(axes.flat, configs):
    ax.plot(data, color=color, lw=2)
    ax.set_title(title); ax.set_xlabel('Step'); ax.set_ylabel('Loss')

plt.suptitle('Training Loss Failure Mode Patterns — Know These Cold', fontsize=13)
plt.tight_layout(); plt.show()

print('Diagnostic actions:')
print('  Oscillating:   reduce LR by 5–10x')
print('  NaN crash:     add gradient clipping (max_norm=1.0), check input data for inf/nan')
print('  Stuck high:    increase model capacity, check feature quality, more epochs')
print('  Healthy:       save this checkpoint and run on test set')

In [ ]:
# ── CELL 9: Practice ──────────────────────────────────────────────────────────
#
# SCENARIO: You're reviewing a colleague's training run for a price prediction
# model (predicting Dubai apartment rent). They share their loss curves and
# training config. Diagnose and fix 3 problems.

np.random.seed(42)

epochs = np.arange(100)

# Problem 1: Colleague used LR=1.0 — classic oscillation
train_p1 = 2.0 + 0.5 * np.sin(epochs * 0.5) + np.random.randn(100) * 0.3
val_p1   = 2.1 + 0.5 * np.sin(epochs * 0.5 + 0.5) + np.random.randn(100) * 0.3

# Problem 2: Colleague forgot validation — overfit badly
train_p2 = 2.0 * np.exp(-epochs / 15) + 0.05 + np.random.randn(100) * 0.02
val_p2   = 2.0 * np.exp(-epochs / 15) + 0.05 + epochs * 0.02 + np.random.randn(100) * 0.05

# Problem 3: Used MSE loss on a dataset with extreme price outliers (villas)
# Model is technically converging but high test MAPE because outliers dominate
train_p3 = 1.5 * np.exp(-epochs / 20) + 0.3 + np.random.randn(100) * 0.04
val_p3   = 1.5 * np.exp(-epochs / 20) + 0.5 + np.random.randn(100) * 0.06

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (tr, vl, title) in zip(axes, [
    (train_p1, val_p1, 'Run 1: LR=1.0'),
    (train_p2, val_p2, 'Run 2: No early stopping'),
    (train_p3, val_p3, 'Run 3: Wrong loss function'),
]):
    ax.plot(tr, 'steelblue', lw=2, label='Train')
    ax.plot(vl, 'tomato',    lw=2, label='Val')
    ax.set_title(f'Problem: {title}'); ax.legend()

plt.suptitle('Dubai Rent Model — Diagnose These 3 Training Runs', fontsize=13)
plt.tight_layout(); plt.show()

print('DIAGNOSES AND FIXES:')
print()
print('Run 1 — LR=1.0 (oscillating loss, train≈val but both noisy):')
print('  Diagnosis: learning rate is too high')
print('  Fix:       reduce LR to 1e-3, add cosine LR schedule')
print()
print('Run 2 — No early stopping (train keeps dropping, val diverges):')
print('  Diagnosis: overfitting — model memorizing train set')
print('  Fix:       add early stopping (patience=10), add dropout/weight decay')
print('  Note:      always monitor val_loss, save best checkpoint not last')
print()
print('Run 3 — MSE on outlier-heavy data (converges but poor real-world accuracy):')
print('  Diagnosis: MSE punishes outlier villas (10M AED) so heavily that the')
print('             model over-corrects for them and ignores normal apartments')
print('  Fix:       switch to Huber loss or log-transform the target (log price)')
print('  Production note: always plot residuals by price segment after training')

print()
print('KEY TAKEAWAYS FROM THIS LESSON')
print('━' * 48)
takeaways = [
    '1. All ML learning = forward pass → loss → gradient → update → repeat.',
    '2. Loss function choice matters more than algorithm choice.',
    '3. Gradient = slope of loss surface. We walk downhill (subtract gradient).',
    '4. Mini-batch GD is the production default. Batch size = 32–256.',
    '5. Learning rate is the most important hyperparameter. Start at 1e-3 with Adam.',
    '6. Adam/AdamW is the default optimizer. SGD+momentum for CNNs when tuned.',
    '7. Always use a validation set. Save the best checkpoint, not the last.',
    '8. Early stopping is free anti-overfitting. Use it on every training run.',
    '9. Know the 4 loss curve failure modes: oscillation, explosion, overfit, underfit.',
    '10. In production: log grad norm, GPU util, val_loss — not just train loss.',
]
for t in takeaways:
    print(f'  {t}')